In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import recall_score, f1_score, roc_auc_score, precision_score
from untils import create_ip_mapping_2017, create_graph_data_2017, GraphDataset
from network import ROEN
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader 
import torch
import os
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv("data/TrafficLabelling/Tuesday-WorkingHours.pcap_ISCX.csv", encoding='latin1')
data = data.fillna(method='ffill')
data[' Label'].unique()

array(['BENIGN', 'FTP-Patator', 'SSH-Patator'], dtype=object)

In [3]:
def predict(model, dataloader):
    model.eval()  # Set the model to evaluation mode
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    all_preds = []

    with torch.no_grad():  # Turn off gradient calculation
        for graph_data in dataloader:
            graph_data = graph_data.to(device)

            # Get model predictions
            edge_predictions = model(graphs=[graph_data], seq_len=1)
            
            # Get predicted class for each edge
            _, predicted = torch.max(edge_predictions[0], dim=1)
            
            # Store true labels and predictions
            all_preds.extend(predicted.cpu().numpy())
    return all_preds

In [4]:
# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(data[' Label'])

# Parse the timestamps
data[' Timestamp'] = pd.to_datetime(data[' Timestamp'])

# Create time windows by minute
data[' Timestamp'] = data[' Timestamp'].dt.floor('T')

# Group data by time window
grouped_data = data.groupby(' Timestamp')

# Dynamically build graphs for each time window and store them in a list
graph_data_seq = []
for name, group in grouped_data:
    
    # Dynamically generate IP mapping for each time window
    ip_to_id = create_ip_mapping_2017(group)
    
    # Build graph for the current time window and pass the time window name
    graph_data_seq.append(create_graph_data_2017(group, ip_to_id, label_encoder, time_window=name))
    
# Initialize device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create DataLoaders for training and test sets
test_dataset = GraphDataset(graph_data_seq, device=device)

test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False, num_workers=0)



d:\Creat\ROEN++\ROENnew\untils.py:44: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:264.)
  edge_index = torch.tensor([source_ids, destination_ids], dtype=torch.long)


In [5]:
# Initialize and train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ROEN(node_in_channels=1, 
            edge_in_channels=80, 
            hidden_channels_node=128,
            hidden_channels_edge=128, 
            mlp_hidden_channels=128, 
            num_edge_classes=3).to(device)
model.load_state_dict(torch.load("models-new/2017/model_epoch_150.pth", map_location=device))

<All keys matched successfully>

In [6]:
p = predict(model, test_dataloader)

In [7]:
from collections import Counter
element_counts = Counter(p)

print("元素种类及数量:")
for element, count in element_counts.items():
    print(f"元素 {element}: {count} 个")

print(f"\n总共有 {len(element_counts)} 种不同的元素。")

元素种类及数量:
元素 0: 432116 个
元素 2: 5877 个
元素 1: 7916 个

总共有 3 种不同的元素。
